# Notebook 07 — Evaluación semántica con 500 queries y recuperación cross-dialogue

Confirmación a mayor escala de la comparación **estático vs. dinámico vs. EMA α=0.6** sobre *Dialog2Flow*.

Diferencias respecto de la notebook 05 (version_4):

1. **Recuperación cross-dialogue**: el índice se consulta con *over-fetch* (top-50) y se descartan
   en query time los vecinos del mismo `dialogue_id` que la consulta, conservando los 10 primeros
   restantes. Esto mide directamente el caso de uso de memoria conversacional (recuperar *otras*
   conversaciones) y pone a las tres variantes en igualdad de condiciones.
2. **500 queries**: las 100 originales (seed 142, idénticas a version_4) más 400 nuevas *held-out*
   (seed 542). Las 400 held-out funcionan como confirmación de α=0.6, elegido con las 100 originales.
3. **Variantes**: sólo `estatico`, `dinamico` y `ema_alpha_0_6` (4.500 juicios en total).

El protocolo de juicio LLM (prompt, schema, modelo, temperatura) es idéntico al de version_4.
Los checkpoints van a `resultados/semantic_llm/version_5/`.

In [ ]:
!pip install -q openai gdown faiss-cpu pandas numpy tqdm scipy

## 1. Imports y montaje de Drive

In [ ]:
import os
import json
import time
import gc
import shutil
from pathlib import Path
from getpass import getpass
from datetime import datetime
from zoneinfo import ZoneInfo

import gdown
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from IPython.display import display, Markdown

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('IN_COLAB:', IN_COLAB)

## 2. Configuración

Constantes heredadas de version_4 sin cambios: split (seed 42), muestra original (seed 142),
parámetros de índices, `K_EVAL`, `CONTEXT_WINDOW`, modelo y temperatura.

Flags nuevos de esta versión (editar acá si se quiere otro diseño):

- `FILTER_SAME_DIALOGUE`: filtro cross-dialogue en query time (default `True`).
- `FILTER_EXACT_DUPLICATES`: además filtra vecinos a distancia < `DUP_EPS` (duplicados
  inter-corpus tipo KETOD/SGD). Default `False`: sólo se reporta la tasa, no se filtra.
- `N_QUERIES_HELDOUT`: tamaño de la muestra de confirmación (default 400).

In [ ]:
if IN_COLAB:
    DRIVE_BASE_DIR = Path('/content/drive/MyDrive/Doctorado/Cursos/ANN/TF')
    INPUT_DIR = DRIVE_BASE_DIR / 'data' / 'id_cache_ema_dialog2flow'
    CHECKPOINT_DIR = DRIVE_BASE_DIR / 'resultados' / 'semantic_llm' / 'version_5' / 'checkpoints'
    FINAL_DIR = DRIVE_BASE_DIR / 'resultados' / 'semantic_llm' / 'version_5' / 'final_tables'
else:
    INPUT_DIR = Path('./ann_ema_d2f_inputs')
    CHECKPOINT_DIR = Path('./ann_v5_checkpoints')
    FINAL_DIR = Path('./ann_v5_final_tables')

RETRIEVAL_DIR = CHECKPOINT_DIR / 'retrieval_checkpoints'
JUDGMENT_DIR = CHECKPOINT_DIR / 'judgment_checkpoints'
SPLIT_DIR = CHECKPOINT_DIR / 'split'

for d in [INPUT_DIR, CHECKPOINT_DIR, RETRIEVAL_DIR, JUDGMENT_DIR, SPLIT_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EMBEDDING_NAME = 'dialog2flow-joint-bert-base'
EMBEDDING_LABEL = 'Dialog2Flow'

# Sólo las tres variantes de la comparación confirmatoria.
VARIANTES_A_EVALUAR = ['estatico', 'dinamico', 'ema_alpha_0_6']
INDICES_A_EVALUAR = ['IVF', 'HNSW', 'IVFPQ']

K_EVAL = 10
CONTEXT_WINDOW = 2

# Split heredado de version_4 (mismos archivos de Drive).
N_QUERIES_SPLIT = 10000
RANDOM_STATE = 42
QUERY_SAMPLE_SEED = RANDOM_STATE + 100        # 142: reproduce las 100 originales
N_QUERIES_ORIGINAL = 100
QUERY_SAMPLE_SEED_HELDOUT = RANDOM_STATE + 500  # 542: muestra nueva de confirmación
N_QUERIES_HELDOUT = 400
N_QUERIES_LLM = N_QUERIES_ORIGINAL + N_QUERIES_HELDOUT

# Recuperación cross-dialogue.
FILTER_SAME_DIALOGUE = True
OVERFETCH_K = 50          # candidatos pedidos al índice antes de filtrar
OVERFETCH_K_FALLBACK = 200  # re-búsqueda si tras filtrar quedan < K_EVAL
FILTER_EXACT_DUPLICATES = False
DUP_EPS = 1e-6            # umbral de distancia L2 para marcar duplicados casi exactos

# Índices FAISS: idénticos a version_4.
N_LIST = 4096
N_PROBE = 64
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 256
PQ_M = 32
PQ_NBITS = 8
N_TRAIN_IVF = 100_000

OPENAI_MODEL = 'gpt-4.1-mini'
TEMPERATURE = 0
SLEEP_BETWEEN_CALLS = 0.2

# Para prueba rápida, descomentar:
# N_QUERIES_HELDOUT = 5; N_QUERIES_LLM = 105
# VARIANTES_A_EVALUAR = ['estatico', 'ema_alpha_0_6']
# INDICES_A_EVALUAR = ['HNSW']

print('Variantes:', VARIANTES_A_EVALUAR)
print('Índices:', INDICES_A_EVALUAR)
print('Queries totales:', N_QUERIES_LLM, f'({N_QUERIES_ORIGINAL} originales + {N_QUERIES_HELDOUT} held-out)')
print('Filtro same-dialogue:', FILTER_SAME_DIALOGUE, '| Filtro duplicados exactos:', FILTER_EXACT_DUPLICATES)
print('Juicios LLM previstos:', len(VARIANTES_A_EVALUAR) * len(INDICES_A_EVALUAR) * N_QUERIES_LLM)

## 3. Archivos de Drive (sólo los necesarios para las tres variantes)

In [ ]:
FILES = {
    'dialogs-2.0.pkl': '1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb',
    'ids.npy': '1hWC7nLvSboFHyykjr7VFAEmcvz8re1cY',

    'index_idx_N10000_seed42.npy': '1UECGYQxr7YClV2VpF476u7NrL-xSoR_V',
    'query_idx_N10000_seed42.npy': '1B_pFn9iKXi16CaDcfv1juJ7fUwRqRFb8',

    'embeddings_dialog2flow.npy': '1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW',
    'accumulative_embeddings_dialog2flow.npy': '1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU',
    'ema_embeddings_dialog2flow_alpha_0_6.npy': '1oGqM0cpIkfGacPr3OY-DW-GYrlYMFH4h',
}

def local_path(filename: str) -> Path:
    return INPUT_DIR / filename

def download_file(filename: str, force: bool = False) -> Path:
    if filename not in FILES:
        raise KeyError(f'Archivo no registrado en FILES: {filename}')
    path = local_path(filename)
    if path.exists() and not force:
        return path
    if path.exists() and force:
        path.unlink()
    file_id = FILES[filename]
    url = f'https://drive.google.com/uc?id={file_id}'
    print('Descargando', filename, '| ID:', file_id)
    gdown.download(url, str(path), quiet=False, fuzzy=True)
    if not path.exists():
        raise FileNotFoundError(f'No se pudo descargar {filename} en {path}')
    return path

def variant_to_filename(variante: str) -> str:
    if variante == 'estatico':
        return 'embeddings_dialog2flow.npy'
    if variante == 'dinamico':
        return 'accumulative_embeddings_dialog2flow.npy'
    if variante.startswith('ema_alpha_'):
        tag = variante.replace('ema_alpha_', '')
        return f'ema_embeddings_dialog2flow_alpha_{tag}.npy'
    raise ValueError(f'Variante no reconocida: {variante}')

def get_embedding_path(variante: str) -> Path:
    return download_file(variant_to_filename(variante), force=False)

def safe_name(text: str) -> str:
    return str(text).replace('/', '_').replace(' ', '_')

## 4. Dataset, metadata de diálogo y contexto textual (idéntico a version_4)

In [ ]:
ruta_df = download_file('dialogs-2.0.pkl')
ruta_ids = download_file('ids.npy')

df = pd.read_pickle(ruta_df).reset_index(drop=True)
ids = np.load(ruta_ids)

print('DataFrame:', df.shape)
if len(df) != len(ids):
    print('ADVERTENCIA: len(df) != len(ids)')

# Mapeo vectorizado row_id -> dialogue_id (para el filtro cross-dialogue).
dialogue_of_row = df['dialogue_id'].astype(str).to_numpy()

_df_ordered = df.reset_index().rename(columns={'index': 'row_id'})
_dialogue_groups = {
    dialogue_id: g.sort_values('turn_id')['row_id'].tolist()
    for dialogue_id, g in _df_ordered.groupby('dialogue_id', sort=False)
}
_row_to_pos = {}
for dialogue_id, rows in _dialogue_groups.items():
    for pos, row_id in enumerate(rows):
        _row_to_pos[int(row_id)] = int(pos)

def get_context_rows(row_id: int, window: int = 2):
    dialogue_id = df.loc[row_id, 'dialogue_id']
    rows = _dialogue_groups[dialogue_id]
    pos = _row_to_pos[int(row_id)]
    start = max(0, pos - window)
    return rows[start:pos + 1]

def format_turn(row_id: int) -> str:
    r = df.loc[row_id]
    speaker = r['speaker'] if 'speaker' in df.columns else '?'
    utterance = str(r['utterance']) if 'utterance' in df.columns else str(r.to_dict())
    turn_id = r['turn_id'] if 'turn_id' in df.columns else row_id
    return f'[{turn_id}] {speaker}: {utterance}'

def format_situation(row_id: int, window: int = 2) -> str:
    rows = get_context_rows(row_id, window=window)
    return '\n'.join(format_turn(rid) for rid in rows)

print(format_situation(0, window=CONTEXT_WINDOW))

## 5. Split y muestra de queries: 100 originales + 400 held-out

Las 100 originales se reproducen con el mismo seed (142) sobre el mismo pool de 10.000.
Si existe el archivo guardado por version_4, se verifica la igualdad. Las 400 held-out se
muestrean del pool excluyendo las originales, con seed 542.

In [ ]:
index_idx = np.load(download_file('index_idx_N10000_seed42.npy')).astype('int64')
query_idx_full = np.load(download_file('query_idx_N10000_seed42.npy')).astype('int64')

# --- 100 originales (idénticas a version_4) ---
rng_orig = np.random.default_rng(QUERY_SAMPLE_SEED)
query_sample_original = rng_orig.choice(
    query_idx_full, size=min(N_QUERIES_ORIGINAL, len(query_idx_full)), replace=False
).astype('int64')

# Verificación opcional contra el archivo de version_4 (si está en Drive).
v4_sample_path = (CHECKPOINT_DIR.parent.parent / 'version_4' / 'checkpoints' / 'split'
                  / f'query_sample_N{N_QUERIES_ORIGINAL}_seed{QUERY_SAMPLE_SEED}.npy')
if v4_sample_path.exists():
    v4_sample = np.load(v4_sample_path).astype('int64')
    assert np.array_equal(v4_sample, query_sample_original), \
        'La muestra reproducida NO coincide con la de version_4. Revisar seeds.'
    print('OK: las 100 originales coinciden con el archivo de version_4.')
else:
    print('Archivo de version_4 no encontrado; se confía en la reproducción por seed.')

# --- 400 held-out (nuevas, excluyendo las originales) ---
heldout_path = SPLIT_DIR / f'query_sample_heldout_N{N_QUERIES_HELDOUT}_seed{QUERY_SAMPLE_SEED_HELDOUT}.npy'
if heldout_path.exists():
    query_sample_heldout = np.load(heldout_path).astype('int64')
    print('Held-out cargado:', heldout_path)
else:
    pool = query_idx_full[~np.isin(query_idx_full, query_sample_original)]
    rng_held = np.random.default_rng(QUERY_SAMPLE_SEED_HELDOUT)
    query_sample_heldout = rng_held.choice(
        pool, size=min(N_QUERIES_HELDOUT, len(pool)), replace=False
    ).astype('int64')
    np.save(heldout_path, query_sample_heldout)
    print('Held-out generado y guardado:', heldout_path)

assert len(np.intersect1d(query_sample_original, query_sample_heldout)) == 0

query_sample = np.concatenate([query_sample_original, query_sample_heldout])
query_set_labels = np.array(['original'] * len(query_sample_original)
                            + ['heldout'] * len(query_sample_heldout))

np.save(SPLIT_DIR / f'query_sample_total_N{len(query_sample)}.npy', query_sample)

print('Vectores indexados:', len(index_idx))
print('Queries totales para LLM:', len(query_sample),
      f'({(query_set_labels == "original").sum()} originales, {(query_set_labels == "heldout").sum()} held-out)')

## 6. Recuperación cross-dialogue con over-fetch (checkpointeada)

Para cada query se piden `OVERFETCH_K` candidatos al índice y se descartan los del mismo
`dialogue_id`, conservando los `K_EVAL` primeros restantes (el orden del índice se preserva).
Si tras filtrar quedan menos de `K_EVAL`, se re-busca con `OVERFETCH_K_FALLBACK`.
Se registra cuántos candidatos same-dialogue se descartaron y cuántos vecinos retenidos
están a distancia < `DUP_EPS` (duplicados inter-corpus, sólo reporte por defecto).

In [ ]:
def retrieval_checkpoint_path(variante: str, index_type: str) -> Path:
    return RETRIEVAL_DIR / f'retrieval__{safe_name(variante)}__{safe_name(index_type)}.csv'

def normalize_copy(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype='float32').copy()
    faiss.normalize_L2(x)
    return x

def build_ann_index(index_type: str, index_vectors: np.ndarray, random_state: int = 42):
    n, d = index_vectors.shape
    if index_type == 'IVF':
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFFlat(quantizer, d, N_LIST, faiss.METRIC_L2)
        rng = np.random.default_rng(random_state)
        train_idx = rng.choice(n, size=min(N_TRAIN_IVF, n), replace=False)
        index.train(index_vectors[train_idx])
        index.add(index_vectors)
        index.nprobe = N_PROBE
        return index
    if index_type == 'HNSW':
        index = faiss.IndexHNSWFlat(d, HNSW_M)
        index.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
        index.add(index_vectors)
        index.hnsw.efSearch = HNSW_EF_SEARCH
        return index
    if index_type == 'IVFPQ':
        if d % PQ_M != 0:
            raise ValueError(f'La dimensión {d} no es divisible por PQ_M={PQ_M}')
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFPQ(quantizer, d, N_LIST, PQ_M, PQ_NBITS)
        rng = np.random.default_rng(random_state)
        train_idx = rng.choice(n, size=min(N_TRAIN_IVF, n), replace=False)
        index.train(index_vectors[train_idx])
        index.add(index_vectors)
        index.nprobe = N_PROBE
        return index
    raise ValueError(f'Índice no reconocido: {index_type}')


def filtrar_candidatos(q_row: int, dists, n_rows, k: int):
    """Aplica el filtro cross-dialogue (y opcionalmente de duplicados) preservando el orden.

    Devuelve (kept, n_same_skipped, n_dup_skipped) donde kept es una lista de
    (ann_rank_original, distancia, neighbor_row, near_dup_flag)."""
    q_dlg = dialogue_of_row[q_row]
    kept, n_same, n_dup = [], 0, 0
    for pos in range(len(n_rows)):
        n_row = int(n_rows[pos])
        if n_row < 0:
            continue
        dist = float(dists[pos])
        if FILTER_SAME_DIALOGUE and dialogue_of_row[n_row] == q_dlg:
            n_same += 1
            continue
        near_dup = bool(dist < DUP_EPS)
        if FILTER_EXACT_DUPLICATES and near_dup:
            n_dup += 1
            continue
        kept.append((pos + 1, dist, n_row, near_dup))
        if len(kept) == k:
            break
    return kept, n_same, n_dup


def retrieve_topk_cross_dialogue(variante: str, index_type: str, k: int = K_EVAL) -> pd.DataFrame:
    ckpt_path = retrieval_checkpoint_path(variante, index_type)
    if ckpt_path.exists():
        print(f'Checkpoint encontrado: {ckpt_path}')
        return pd.read_csv(ckpt_path)

    path = get_embedding_path(variante)
    matriz = np.load(path, mmap_mode='r')

    print('=' * 80)
    print('Recuperación cross-dialogue |', variante, '/', index_type)
    print('Archivo:', path.name, '| Shape:', matriz.shape)

    index_vectors = normalize_copy(matriz[index_idx])
    query_vectors = normalize_copy(matriz[query_sample])

    t0 = time.time()
    index = build_ann_index(index_type, index_vectors, random_state=RANDOM_STATE)
    build_time = time.time() - t0

    t1 = time.time()
    distances, local_neighbors = index.search(query_vectors, OVERFETCH_K)
    search_time = time.time() - t1
    neighbor_rows = index_idx[local_neighbors]

    # Primer pase de filtrado; se anotan las queries que quedaron cortas.
    kept_por_query, deficientes = {}, []
    for q_pos, q_row in enumerate(query_sample):
        kept, n_same, n_dup = filtrar_candidatos(int(q_row), distances[q_pos], neighbor_rows[q_pos], k)
        kept_por_query[q_pos] = (kept, n_same, n_dup)
        if len(kept) < k:
            deficientes.append(q_pos)

    if deficientes:
        print(f'Re-búsqueda con k={OVERFETCH_K_FALLBACK} para {len(deficientes)} queries...')
        d2, i2 = index.search(query_vectors[deficientes], OVERFETCH_K_FALLBACK)
        rows2 = index_idx[i2]
        for j, q_pos in enumerate(deficientes):
            kept, n_same, n_dup = filtrar_candidatos(int(query_sample[q_pos]), d2[j], rows2[j], k)
            kept_por_query[q_pos] = (kept, n_same, n_dup)
            if len(kept) < k:
                print(f'ADVERTENCIA: query_order={q_pos} quedó con {len(kept)} vecinos (< {k}).')

    records = []
    for q_pos, q_row in enumerate(query_sample):
        q_row = int(q_row)
        kept, n_same, n_dup = kept_por_query[q_pos]
        for out_rank, (ann_rank, dist, n_row, near_dup) in enumerate(kept, start=1):
            records.append({
                'embedding_name': EMBEDDING_NAME,
                'embedding_base': EMBEDDING_LABEL,
                'variant': variante,
                'index_type': index_type,
                'query_order': int(q_pos),
                'query_set': str(query_set_labels[q_pos]),
                'query_row': q_row,
                'neighbor_rank': int(out_rank),
                'ann_rank_prefilter': int(ann_rank),
                'neighbor_row': int(n_row),
                'distance': dist,
                'near_duplicate': bool(near_dup),
                'n_same_dialogue_skipped': int(n_same),
                'n_duplicates_skipped': int(n_dup),
                'query_utterance': str(df.loc[q_row, 'utterance']),
                'neighbor_utterance': str(df.loc[n_row, 'utterance']),
                'query_context': format_situation(q_row, window=CONTEXT_WINDOW),
                'neighbor_context': format_situation(n_row, window=CONTEXT_WINDOW),
                'build_time_s': build_time,
                'search_time_s': search_time,
            })

    df_part = pd.DataFrame(records)
    tmp_path = ckpt_path.with_suffix('.tmp.csv')
    df_part.to_csv(tmp_path, index=False)
    os.replace(tmp_path, ckpt_path)
    print('Checkpoint guardado:', ckpt_path)

    del matriz, index_vectors, query_vectors, index, distances, local_neighbors, neighbor_rows
    gc.collect()
    return df_part


def run_all_retrievals_checkpointed():
    parts = []
    total = len(VARIANTES_A_EVALUAR) * len(INDICES_A_EVALUAR)
    done = 0
    for variante in VARIANTES_A_EVALUAR:
        for index_type in INDICES_A_EVALUAR:
            done += 1
            print(f'\n[{done}/{total}] {variante} / {index_type}')
            parts.append(retrieve_topk_cross_dialogue(variante, index_type, k=K_EVAL))
    return pd.concat(parts, ignore_index=True)

df_retrieval = run_all_retrievals_checkpointed()
print('Recuperaciones totales:', df_retrieval.shape)

# Sanity check del filtro: ningún vecino retenido puede compartir diálogo con su query.
_same = (df_retrieval['query_row'].map(lambda r: dialogue_of_row[int(r)]).values
         == df_retrieval['neighbor_row'].map(lambda r: dialogue_of_row[int(r)]).values)
print('Pares same-dialogue retenidos (debe ser 0 con filtro ON):', int(_same.sum()))
print('Tasa de vecinos near-duplicate retenidos:', round(float(df_retrieval['near_duplicate'].mean()) * 100, 2), '%')
print('Promedio de candidatos same-dialogue descartados por query:',
      round(float(df_retrieval.groupby(['variant', 'index_type', 'query_order'])['n_same_dialogue_skipped'].first().mean()), 2))
display(df_retrieval.head())

## 7. Inspección rápida de una recuperación

In [ ]:
def mostrar_vecinos(df_retrieval, variante='ema_alpha_0_6', index_type='HNSW', query_order=0, top_n=10):
    sub = df_retrieval[
        (df_retrieval['variant'] == variante) &
        (df_retrieval['index_type'] == index_type) &
        (df_retrieval['query_order'] == query_order)
    ].sort_values('neighbor_rank').head(top_n)
    if sub.empty:
        print('No hay registros para ese filtro.')
        return
    first = sub.iloc[0]
    display(Markdown(f'### Consulta `{query_order}` ({first["query_set"]}) — {variante} / {index_type}'))
    display(Markdown('**Situación de consulta:**'))
    print(first['query_context'])
    display(sub[['neighbor_rank', 'ann_rank_prefilter', 'distance', 'near_duplicate', 'neighbor_row', 'neighbor_utterance']])

mostrar_vecinos(df_retrieval, variante='ema_alpha_0_6', index_type='HNSW', query_order=0)

## 8. Cliente OpenAI

In [ ]:
from openai import OpenAI

if 'OPENAI_API_KEY' not in os.environ or not os.environ['OPENAI_API_KEY']:
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
print('Cliente OpenAI configurado.')

## 9. Juez LLM: prompt y schema (idénticos a version_4)

In [ ]:
EVALUATION_SCHEMA = {
    'name': 'dialog_retrieval_evaluation',
    'schema': {
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'evaluations': {
                'type': 'array',
                'minItems': K_EVAL,
                'maxItems': K_EVAL,
                'items': {
                    'type': 'object',
                    'additionalProperties': False,
                    'properties': {
                        'rank': {'type': 'integer', 'minimum': 1, 'maximum': K_EVAL},
                        'semantic_similarity': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'functional_similarity': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'memory_usefulness': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'overall_similarity': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'brief_reason': {'type': 'string'},
                    },
                    'required': [
                        'rank',
                        'semantic_similarity',
                        'functional_similarity',
                        'memory_usefulness',
                        'overall_similarity',
                        'brief_reason',
                    ],
                },
            }
        },
        'required': ['evaluations'],
    },
    'strict': True,
}

SYSTEM_PROMPT = (
    'You are an expert evaluator of task-oriented dialogue retrieval.\n'
    'Your task is to judge whether retrieved dialogue situations are semantically and functionally similar to a query dialogue situation.\n'
    'Focus on task-oriented dialogue behavior, not only lexical overlap.\n'
    'Use the 1-5 scale consistently.\n'
    'Return only valid JSON following the schema.'
)

def build_judge_prompt(group: pd.DataFrame) -> str:
    group = group.sort_values('neighbor_rank')
    first = group.iloc[0]
    lines = []
    lines.append('Evaluate whether each retrieved neighbor is similar to the query situation.')
    lines.append('')
    lines.append('Scoring scale:')
    lines.append('1 = unrelated')
    lines.append('2 = weak or superficial relation')
    lines.append('3 = partial similarity')
    lines.append('4 = clear semantic/functional similarity')
    lines.append('5 = highly equivalent dialogue situations')
    lines.append('')
    lines.append('QUERY SITUATION:')
    lines.append(first['query_context'])
    lines.append('')
    lines.append('RETRIEVED NEIGHBORS:')
    for _, r in group.iterrows():
        lines.append('')
        lines.append(f"Neighbor rank {int(r['neighbor_rank'])}:")
        lines.append(r['neighbor_context'])
    lines.append('')
    lines.append(f'Return one evaluation object for each neighbor rank from 1 to {K_EVAL}.')
    return '\n'.join(lines)

def evaluate_group_with_llm(group: pd.DataFrame, model: str = OPENAI_MODEL) -> dict:
    user_prompt = build_judge_prompt(group)
    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        response_format={
            'type': 'json_schema',
            'json_schema': EVALUATION_SCHEMA,
        },
    )
    return json.loads(response.choices[0].message.content)

_test_key, _test_group = next(iter(df_retrieval.groupby(['embedding_base', 'variant', 'index_type', 'query_order'])))
print(build_judge_prompt(_test_group)[:2500])

## 10. Juicios LLM con checkpoints durables (idéntico a version_4)

In [ ]:
def judgment_checkpoint_path(variante: str, index_type: str) -> Path:
    return JUDGMENT_DIR / f'judgments__{safe_name(variante)}__{safe_name(index_type)}.jsonl'

def judgment_key(variante: str, index_type: str, query_order: int, query_row: int) -> str:
    return '|'.join([
        EMBEDDING_LABEL,
        str(variante),
        str(index_type),
        str(query_order),
        str(query_row),
        OPENAI_MODEL,
    ])

def load_done_judgments(path: Path) -> set:
    done = set()
    if not path.exists():
        return done
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                done.add(rec['config_key'])
            except Exception:
                pass
    return done

def append_jsonl_durable(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())

def run_llm_for_config(df_config: pd.DataFrame, variante: str, index_type: str):
    out_path = judgment_checkpoint_path(variante, index_type)
    done = load_done_judgments(out_path)
    groups = list(df_config.groupby('query_order', sort=False))
    print('=' * 80)
    print('LLM config:', variante, '/', index_type)
    print('Queries:', len(groups), '| Ya evaluadas:', len(done))
    for query_order, group in tqdm(groups):
        group = group.sort_values('neighbor_rank')
        query_row = int(group.iloc[0]['query_row'])
        config_key = judgment_key(variante, index_type, int(query_order), query_row)
        if config_key in done:
            continue
        try:
            result = evaluate_group_with_llm(group, model=OPENAI_MODEL)
            record = {
                'config_key': config_key,
                'embedding_base': EMBEDDING_LABEL,
                'variant': variante,
                'index_type': index_type,
                'query_order': int(query_order),
                'query_set': str(group.iloc[0]['query_set']),
                'query_row': query_row,
                'model': OPENAI_MODEL,
                'evaluations': result['evaluations'],
            }
            append_jsonl_durable(out_path, record)
            done.add(config_key)
            time.sleep(SLEEP_BETWEEN_CALLS)
        except Exception as e:
            print('Error en', variante, index_type, query_order, '->', repr(e))
            time.sleep(2)
    print('Finalizado:', variante, '/', index_type)

def run_all_llm_checkpointed(df_retrieval: pd.DataFrame):
    total = len(VARIANTES_A_EVALUAR) * len(INDICES_A_EVALUAR)
    done_cfg = 0
    for variante in VARIANTES_A_EVALUAR:
        for index_type in INDICES_A_EVALUAR:
            done_cfg += 1
            print(f'\n[{done_cfg}/{total}] Evaluación LLM: {variante} / {index_type}')
            df_config = df_retrieval[
                (df_retrieval['variant'] == variante) &
                (df_retrieval['index_type'] == index_type)
            ].copy()
            if df_config.empty:
                print('Sin retrieval para esta configuración. Se omite.')
                continue
            run_llm_for_config(df_config, variante, index_type)

run_all_llm_checkpointed(df_retrieval)

## 11. Progreso

In [ ]:
progress_rows = []
for variante in VARIANTES_A_EVALUAR:
    for index_type in INDICES_A_EVALUAR:
        path = judgment_checkpoint_path(variante, index_type)
        done = load_done_judgments(path)
        progress_rows.append({
            'variant': variante,
            'index_type': index_type,
            'expected_queries': len(query_sample),
            'done_queries': len(done),
            'pending_queries': max(0, len(query_sample) - len(done)),
            'complete': len(done) >= len(query_sample),
        })

progress_df = pd.DataFrame(progress_rows)
display(progress_df)
print('Pendientes totales:', progress_df['pending_queries'].sum())

## 12. Consolidación de juicios

In [ ]:
def read_all_judgments() -> pd.DataFrame:
    records = []
    files = sorted(JUDGMENT_DIR.glob('judgments__*.jsonl'))
    print('Archivos de juicios encontrados:', len(files))
    for path in files:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                for ev in rec['evaluations']:
                    records.append({
                        'embedding_base': rec['embedding_base'],
                        'variant': rec['variant'],
                        'index_type': rec['index_type'],
                        'query_order': rec['query_order'],
                        'query_set': rec.get('query_set', ''),
                        'query_row': rec['query_row'],
                        'model': rec['model'],
                        **ev,
                    })
    if not records:
        return pd.DataFrame()
    scores = pd.DataFrame(records).rename(columns={'rank': 'neighbor_rank'})
    return scores

df_scores_raw = read_all_judgments()
# Dedupe defensivo: si un jsonl tuviera líneas duplicadas (crash + retry), no contar dos veces.
_antes = len(df_scores_raw)
df_scores_raw = df_scores_raw.drop_duplicates(
    subset=['embedding_base', 'variant', 'index_type', 'query_order', 'neighbor_rank'], keep='last')
if len(df_scores_raw) != _antes:
    print('Juicios duplicados removidos:', _antes - len(df_scores_raw))
print('Scores raw:', df_scores_raw.shape)

merge_cols = ['embedding_base', 'variant', 'index_type', 'query_order', 'query_row', 'neighbor_rank']
df_scores = df_scores_raw.drop(columns=['query_set']).merge(
    df_retrieval, on=merge_cols, how='left', validate='many_to_one',
)
print('Scores enriquecidos:', df_scores.shape)
display(df_scores.head())

## 13. Métricas y tests

- MSS@10 con SD, SE e IC95 por variante e índice (500 queries).
- Wilcoxon signed-rank pareado para los tres contrastes (EMA 0.6 vs dinámico, EMA 0.6 vs estático,
  dinámico vs estático), sobre las 500 queries y, como **test confirmatorio**, sobre las 400 held-out.
- W/T/L por contraste.

In [ ]:
from scipy.stats import wilcoxon

if df_scores.empty:
    raise ValueError('No hay juicios consolidados.')

query_metrics = (
    df_scores
    .groupby(['embedding_base', 'variant', 'index_type', 'query_set', 'query_order'], as_index=False)
    .agg(
        semantic_score_at10=('semantic_similarity', 'mean'),
        functional_score_at10=('functional_similarity', 'mean'),
        memory_score_at10=('memory_usefulness', 'mean'),
        mss_at10=('overall_similarity', 'mean'),
    )
)

summary_metrics = (
    query_metrics
    .groupby(['embedding_base', 'variant', 'index_type'], as_index=False)
    .agg(
        n_queries=('query_order', 'nunique'),
        mean_mss_at10=('mss_at10', 'mean'),
        sd_mss_at10=('mss_at10', 'std'),
    )
)
summary_metrics['se_mss_at10'] = summary_metrics['sd_mss_at10'] / np.sqrt(summary_metrics['n_queries'])
summary_metrics['ci95_mss_at10'] = 1.96 * summary_metrics['se_mss_at10']
for col in summary_metrics.select_dtypes(include=['float']).columns:
    summary_metrics[col] = summary_metrics[col].round(3)
print('Resumen (500 queries):')
display(summary_metrics)

paired = query_metrics.pivot_table(
    index=['index_type', 'query_set', 'query_order'],
    columns='variant', values='mss_at10',
).reset_index()

EPS = 1e-9
CONTRASTES = [
    ('ema_alpha_0_6', 'dinamico'),
    ('ema_alpha_0_6', 'estatico'),
    ('dinamico', 'estatico'),
]

rows = []
for scope, scope_df in [('full_500', paired), ('heldout_400', paired[paired['query_set'] == 'heldout'])]:
    for index_type, g in scope_df.groupby('index_type', sort=False):
        for a, b in CONTRASTES:
            if a not in g.columns or b not in g.columns:
                continue
            sub = g.dropna(subset=[a, b])
            delta = sub[a] - sub[b]
            try:
                stat, p = wilcoxon(delta)
            except Exception:
                stat, p = np.nan, np.nan
            rows.append({
                'scope': scope,
                'index_type': index_type,
                'contraste': f'{a} - {b}',
                'n': len(sub),
                'mean_delta': round(float(delta.mean()), 4),
                'wilcoxon_p': p,
                'wins': int((delta > EPS).sum()),
                'ties': int((delta.abs() <= EPS).sum()),
                'losses': int((delta < -EPS).sum()),
            })

tests_df = pd.DataFrame(rows)
tests_df['wilcoxon_p'] = tests_df['wilcoxon_p'].map(lambda p: float(f'{p:.4g}') if pd.notna(p) else p)
print('Tests pareados (orden de índices: IVF, HNSW, IVFPQ):')
display(tests_df)

print('Confirmatorio held-out (EMA 0.6 vs dinámico):')
display(tests_df[(tests_df.scope == 'heldout_400') & (tests_df.contraste == 'ema_alpha_0_6 - dinamico')])

## 14. Inspección cualitativa con scores

In [ ]:
def mostrar_vecinos_con_scores(df_scores, variante='ema_alpha_0_6', index_type='HNSW', query_order=0, top_n=10):
    sub = df_scores[
        (df_scores['variant'] == variante) &
        (df_scores['index_type'] == index_type) &
        (df_scores['query_order'] == query_order)
    ].sort_values('neighbor_rank').head(top_n)
    if sub.empty:
        print('No hay registros para ese filtro.')
        return
    first = sub.iloc[0]
    display(Markdown(f'### Consulta `{query_order}` — {variante} / {index_type}'))
    display(Markdown('**Situación de consulta:**'))
    print(first['query_context'])
    cols = ['neighbor_rank', 'distance', 'overall_similarity', 'semantic_similarity',
            'functional_similarity', 'memory_usefulness', 'neighbor_utterance', 'brief_reason']
    display(sub[cols])

mostrar_vecinos_con_scores(df_scores, variante='ema_alpha_0_6', index_type='HNSW', query_order=0)

## 15. Exportación de resultados

In [ ]:
fecha_hora_arg = datetime.now(ZoneInfo('America/Argentina/Buenos_Aires')).strftime('%Y%m%d_%H%M%S_ARG')

outputs = {
    f'retrieval_top10_crossdlg_{fecha_hora_arg}.csv': ('retrieval_top10_crossdlg_latest.csv', df_retrieval),
    f'llm_scores_pairs_crossdlg_{fecha_hora_arg}.csv': ('llm_scores_pairs_crossdlg_latest.csv', df_scores),
    f'llm_query_metrics_crossdlg_{fecha_hora_arg}.csv': ('llm_query_metrics_crossdlg_latest.csv', query_metrics),
    f'llm_summary_crossdlg_{fecha_hora_arg}.csv': ('llm_summary_crossdlg_latest.csv', summary_metrics),
    f'wilcoxon_tests_crossdlg_{fecha_hora_arg}.csv': ('wilcoxon_tests_crossdlg_latest.csv', tests_df),
    f'progress_crossdlg_{fecha_hora_arg}.csv': ('progress_crossdlg_latest.csv', progress_df),
}

for filename, (latest_name, df_out) in outputs.items():
    path = FINAL_DIR / filename
    df_out.to_csv(path, index=False)
    shutil.copy(path, FINAL_DIR / latest_name)
    print('Escrito:', path)

# Mini tabla LaTeX para el paper (MSS@10 ± SD por variante e índice, 500 queries).
pivot_tex = summary_metrics.pivot_table(index='variant', columns='index_type', values='mean_mss_at10')
sd_tex = summary_metrics.pivot_table(index='variant', columns='index_type', values='sd_mss_at10')
orden_v = [v for v in ['estatico', 'dinamico', 'ema_alpha_0_6'] if v in pivot_tex.index]
orden_i = [i for i in ['IVF', 'HNSW', 'IVFPQ'] if i in pivot_tex.columns]
tex_df = pd.DataFrame({
    idx: [f'{pivot_tex.loc[v, idx]:.3f} $\\pm$ {sd_tex.loc[v, idx]:.3f}' for v in orden_v]
    for idx in orden_i
}, index=orden_v)
latex_path = FINAL_DIR / f'table_mss_crossdlg_{fecha_hora_arg}.tex'
latex_path.write_text(tex_df.to_latex(escape=False), encoding='utf-8')
shutil.copy(latex_path, FINAL_DIR / 'table_mss_crossdlg_latest.tex')
print('Escrito:', latex_path)

## 16. Interpretación rápida

- `summary_metrics`: si los IC95 de EMA 0.6 y dinámico no se solapan contra estático, el
  resultado central queda confirmado bajo protocolo cross-dialogue.
- `tests_df` (scope `heldout_400`, contraste `ema_alpha_0_6 - dinamico`): es el test
  **confirmatorio** de α=0.6 sobre queries no usadas para elegirlo. Reportar este p-value en el paper.
- La tasa de `near_duplicate` retenidos cuantifica los duplicados inter-corpus (KETOD/SGD);
  si resulta alta, considerar `FILTER_EXACT_DUPLICATES = True` y re-correr (los checkpoints de
  retrieval deben borrarse en ese caso, los juicios quedan invalidados).
- Poder estadístico aproximado (SD de Δ ≈ 0.49 en version_4): con n=500 se detectan efectos
  ≥ 0.06; con n=400, ≥ 0.07.